
## Introduction
---
As a machine learning engineer working for a social media company, your responsibility is to assess whether songs shared on the platform are suitable for children. Since evaluating every song using large language models (LLMs) is computationally expensive, an alternative approach using Retrieval-Augmented Generation (RAG) is introduced.

RAG works by combining a retrieval component, which finds relevant information (such as embeddings of previously answered questions about content safety), with a generation component that uses this retrieved context to make predictions about new songs. This method allows the system to scale efficiently while still ensuring that each song is properly evaluated for child safety, without needing to run a full LLM inference for every single case.

---

### Import Libraries

In [1]:
import torch
import numpy as np
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from transformers import BertTokenizer, AutoModelForSequenceClassification


### Helper Function

In [2]:
def tsne_plot(data, plot):
    
    tsne = TSNE(n_components=3, random_state=42, perplexity=(50,data.shape[0]-1))
    data_3d = tsne.fit_transform(data)
    
    fig = plt.figure(figsize=(8,8))
    ax = fig.add_subplot(111, projection='3d')
    
    color = plt.cm.rainbow(np.linspace(0,1,len(data_3d)))
    
    for idx, point, in zip(range(len(data_3d)), data_3d):
        ax.scatter(point[0], point[1], point[2], color= color[idx], label = f'{plot} {idx+1}')
    
    ax.set_xlabel("TSNE component 1")
    ax.set_ylabel("TSNE Component 2")
    ax.set_zlabel("TSNE Component 3")
    
    plt.title("3D-TSNE Visualization of "+ plot+ " Embeddings")
    plt.legend(title = plot + "index", bbox_to_anchor = (1.05,1), loc = 'upper left')
    plt.show()

### Tokenizer and Model

In [3]:
model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)

We consider the input as a list of tuple

In [4]:
# Input text to get embeddings for
input_text = [("This is an example sentence for BERT embeddings.", "How do you like it "),("There are other models")]

We use **batch_encode_plus** method for tokenizing the text. It automatically handles the padding and truncation to ensure uniformity in input length, which is difficult for batch processing like BERT models.

In [5]:
tokenized_input = tokenizer(input_text, padding=True, truncation=True, add_special_tokens=True)
tokenized_input

{'input_ids': [[101, 2023, 2003, 2019, 2742, 6251, 2005, 14324, 7861, 8270, 4667, 2015, 1012, 102, 2129, 2079, 2017, 2066, 2009, 102], [101, 2045, 2024, 2060, 4275, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}

In [7]:
tokenized_input['input_ids'][0]

[101,
 2023,
 2003,
 2019,
 2742,
 6251,
 2005,
 14324,
 7861,
 8270,
 4667,
 2015,
 1012,
 102,
 2129,
 2079,
 2017,
 2066,
 2009,
 102]

In [8]:
# confirm tokenization by verification
decode_input = tokenizer.decode(tokenized_input['input_ids'][0])
decode_input

'[CLS] this is an example sentence for bert embeddings. [SEP] how do you like it [SEP]'

In [9]:
print(f"length {len(decode_input.split())}")

length 16


In [ ]:
## Define Device